# Historical Price & Return Analysis Framework
### Consecutive streak detection · Cumulative threshold analysis · Rare-event probabilities · Interactive Plotly dashboard

All helpers live in [`src/tools/price_return.py`](../src/tools/price_return.py);
this notebook configures a `Params` object and calls them.

The rare-event table at the end is **interactive** - select holding periods and
set the probability bounds without re-running the cell.

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with
# `pip install -e .`, so `from src.tools...` resolves from any working
# directory. On Colab the same editable install is done here against the
# copy of the repo on Drive.
import os

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    !pip install -q -e /content/drive/MyDrive/github/quant_dev

print(f'Running in Colab: {IN_COLAB}')

In [ ]:
import pandas as pd

from src.tools.price_return import (
    Params,
    # data
    load_price_data, add_rolling_stats, show_latest_snapshot, daily_returns_series,
    # streak & cumulative analysis
    detect_streaks, summarize_streaks, analyze_cumulative, summarize_cumulative,
    # rare-event probabilities
    build_historical_analysis, filter_low_probability,
    low_probability_view, format_probability_table, interactive_low_probability,
    # charts
    plot_rolling_average, plot_rolling_volatility,
    plot_price_and_returns, plot_return_distribution,
    plot_streak_counts, plot_streak_frequency, plot_streak_timeline,
    plot_cumulative_heatmap, plot_cumulative_counts,
    # export
    export_tables,
)

## Parameters

`Params` carries every tunable value. Fields left out keep their defaults - see
`Params` in [`src/tools/price_return.py`](../src/tools/price_return.py).
Edit here and re-run from this cell down; nothing is bound at import time.

In [ ]:
P = Params(
    # ── Price data ───────────────────────────────────────────────────────
    data_source = 'simulated',     # 'simulated' or 'yahoo'
    ticker      = 'AAPL',
    start_date  = '2020-01-01',
    # end_date defaults to today

    # Simulation-only knobs (ignored when data_source == 'yahoo')
    random_seed     = 42,
    sim_drift       = 0.04,        # mean daily return (%)
    sim_vol         = 1.2,         # daily return std dev (%)
    sim_start_price = 150.0,

    # ── Rolling / streak analysis ────────────────────────────────────────
    trade_days     = 250,          # trading days per year
    win_threshold  =  0.5,         # min daily return (%) to count as a 'win'
    loss_threshold = -0.5,         # max daily return (%) to count as a 'loss'
    windows        = [2, 3, 5],    # consecutive-day streak windows
    cum_thresholds = [0.5, 1.0, 2.0],   # cumulative return thresholds (%)
    # roll_windows defaults to [1, 2, 3, 4, 5, 10, trade_days]

    # ── Charts ───────────────────────────────────────────────────────────
    chart_windows   = [1, 2, 3, 4, 5, 10],   # holding periods drawn on rolling-stat charts
    timeline_window = 2,                     # streak window highlighted on the timeline

    # ── Historical (rare-event) analysis ─────────────────────────────────
    return_thresholds = [0.05, 0.045, 0.04, 0.035, 0.03, 0.025, 0.02,
                         0.015, 0.01, 0.005, 0.002, 0.001, 0.0001],   # 0.01 = a 1% return
    lookback_years    = [2, 5],                        # lookback periods (years)
    streak_days       = [1, 2, 3, 4, 5, 10, 20, 30],   # holding periods (trading days)
    prob_max          = 0.10,      # "low probability" cutoff
    prob_min          = 1e-4,      # drop events never seen in the lookback window
)
P

# Analysis

## Data & rolling statistics

In [ ]:
# Q: Does the series load cleanly, and over what period?
# Check the printed row count and date range, and that `return_pct` is in
# percent (0.5 == +0.5%). The rolling columns are NaN for the first
# `trade_days` rows - that warm-up is expected, not a data error.
df = load_price_data(P)
df = add_rolling_stats(df, P)
df.head()

In [ ]:
# Q: Where do return and risk sit right now, versus the trailing year?
# The annualized pair summarises the last `trade_days` days; the daily pair is
# the per-day mean and std behind it. Compare the daily std against
# win/loss_threshold in P - if they are far apart, the streak counts below are
# driven by that mismatch rather than by any genuine trend.
_ = show_latest_snapshot(df, P)

In [ ]:
# Q: Are average return and volatility stable, or shifting between regimes?
# Average-return chart: lines should scale with holding period (a 10d return is
#   roughly 10x a 1d return). Where they converge or cross, the trend is breaking.
# Volatility chart: the dashed annualized line is the 1d std scaled by
#   sqrt(trade_days). Where the 10d line sits ABOVE it, moves are trending
#   (positively autocorrelated); below it, they are mean-reverting.
plot_rolling_average(df, P).show()
plot_rolling_volatility(df, P).show()

## Streaks & cumulative thresholds

In [ ]:
# Q: How often do wins and losses cluster into consecutive runs?
# One row per streak window; a streak means EVERY day in the window cleared
# win_threshold (or loss_threshold).
# Look for: asymmetry between win and loss frequency (directional bias), and how
# fast frequency decays as the window grows - a slow decay means runs persist
# longer than independent daily moves would imply.
streaks   = detect_streaks(df, P)
streak_df = summarize_streaks(df, streaks, P)
streak_df

In [ ]:
# Q: How often does the COMPOUNDED move over a window clear each threshold?
# Unlike a streak, individual days may be negative as long as the total clears.
# Look for: hit-rate rising both with window length and with lower thresholds.
# Read alongside the streak table above to separate "one big move" from
# "a sustained run".
cum_analysis = analyze_cumulative(df, P)
cum_df       = summarize_cumulative(cum_analysis, P)
cum_df

In [ ]:
# Q: What does the raw series look like, and is its distribution well-behaved?
# Price/returns chart: locate volatility clusters and drawdowns. The dashed lines
#   are the win/loss thresholds, so the density of bars beyond them previews the
#   streak counts.
# Distribution chart: check symmetry and tail weight. Fat tails or visible skew
#   mean the rare-event probabilities below are driven by a handful of days.
plot_price_and_returns(df, P).show()
plot_return_distribution(df, P).show()

In [ ]:
# Q: Where do the streaks fall, and are they biased up or down?
# Counts vs frequency: the same data absolute and normalized - use frequency when
#   comparing across samples of different length.
# Timeline: shows WHEN streaks happened. Look for clustering in particular periods
#   rather than an even spread; clustering means the unconditional probabilities
#   above understate the risk of hitting a run inside a bad regime.
plot_streak_counts(streaks, P).show()
plot_streak_frequency(df, streaks, P).show()
plot_streak_timeline(df, streaks, P).show()

In [ ]:
# Q: Which window / threshold combinations are actually achievable?
# Heatmap: read ACROSS a row to see how fast the hit-rate falls as the threshold
#   rises, and DOWN a column to see what holding longer buys you.
# Bar chart: the same counts grouped by threshold - use it to pick a target that
#   is both large enough to matter and frequent enough to act on.
plot_cumulative_heatmap(cum_analysis, P).show()
plot_cumulative_counts(cum_analysis, P).show()

## Historical rare-event probabilities

In [ ]:
# Q: Over each lookback (2y and 5y), how likely is a move of each size?
# Builds every threshold x holding period x lookback combination, for both event
# types: `consecutive` (every day clears the threshold) and `cumulative` (the
# summed move clears it).
# Check `n_obs` first: where it is below n_years * trade_days the lookback is
# truncated by available history, and the two lookbacks overlap heavily.
pct_change = daily_returns_series(df)
df_his     = build_historical_analysis(pct_change, P)

print(f'Historical analysis: {len(df_his)} rows   |   '
      f'{len(P.return_thresholds)} thresholds x {len(P.streak_days)} holding periods x '
      f'{len(P.lookback_years)} lookbacks x 4 event types')
for n_years in P.lookback_years:
    n_obs = df_his.loc[df_his['n_years'] == n_years, 'n_obs'].iloc[0]
    print(f'  {n_years}y lookback: {n_obs} trading days available '
          f'(of {n_years * P.trade_days} requested)')
df_his.head()

In [ ]:
# Q: Which events are rare enough to be worth pricing?
# Keeps events below prob_max that occurred at least once (prob > prob_min) - an
# event that never happened is unpriceable, not attractive.
# `last_occurred` is the reality check: a "rare" event that last fired last week
# is a very different proposition from one last seen years ago.
# filter_low_probability handles one holding period at a time, so sweep streak_days.
df_low = pd.concat([filter_low_probability(df_his, n_days, P) for n_days in P.streak_days],
                   ignore_index=True)

print(f'{len(df_low)} of {len(df_his)} events are rarer than {P.prob_max:.0%} '
      f'and occurred at least once in their lookback window')
df_low.head(20)

In [ ]:
# Q: Where do the rare events concentrate?
# Counts low-probability events per holding period, split by lookback and event
# type. Expect `consecutive` to dominate at LONG holding periods (runs are hard to
# sustain) and `cumulative` at SHORT ones (little time to accumulate a big move).
# A row that is 0 across both lookbacks means nothing rare lives at that period.
pivot_low = (df_low
             .pivot_table(index='n_days', columns=['n_years', 'change_type'],
                          values='prob', aggfunc='count', fill_value=0)
             .astype(int))
print('Count of low-probability events by holding period')
pivot_low

In [ ]:
# Q: For a holding period, which thresholds are rare - and when were they last seen?
# Interactive: change a control and the table refilters in place, no re-run needed.
#   Streak days - holding period to inspect, one at a time
#   Change type - 'consecutive' (every day clears the threshold), 'cumulative'
#                 (the summed move clears it), or 'All' for both
#   prob_max    - upper bound; lower it to keep only the rarer events
#   prob_min    - lower bound; raise it to drop the near-never events
# Compare the 2y and 5y rows for one threshold: a large gap means the estimate is
# regime-dependent, and the shorter lookback may be fitting recent conditions.
#
# Needs ipywidgets (preinstalled on Colab; `pip install ipywidgets` locally).
# Without it this prints a static table using the P defaults instead.
_ = interactive_low_probability(df_his, P, n_days=3)

In [ ]:
# Non-interactive equivalent, for exporting or when ipywidgets is unavailable.
# `low_probability_view` accepts one holding period or a list of them.
drill = low_probability_view(df_his, 3, P)
print(f'{len(drill)} low-probability events for a 3-day holding period')
format_probability_table(drill)

## Export

In [ ]:
# Writes every table above to CSV in the working directory (Drive when on Colab).
# `rolling_stats.csv` is the full per-day frame; the others are the summaries.
export_tables({
    'streak_summary.csv':               streak_df,
    'cumulative_summary.csv':           cum_df,
    'rolling_stats.csv':                df,
    'historical_analysis.csv':          df_his,
    'historical_analysis_low_prob.csv': df_low,
})